# Train hadith-scholar Model (Google Colab)

Fine-tune Phi-4-mini on hadith/Quran data using Unsloth + QLoRA.
This notebook runs on Google Colab's free T4 GPU.

**Prerequisites:**
- Upload `data/train.jsonl` to Colab (generated locally by `scripts/prepare_training_data.py`)

**Runtime:** Make sure you select **GPU** runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Install Unsloth (Colab-optimized)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

In [ ]:
# Cell 2: Upload training data
from google.colab import files
import os

if not os.path.exists("train.jsonl"):
    print("Upload train.jsonl (generated by: python3 scripts/prepare_training_data.py)")
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print("train.jsonl already exists")

# Count and verify examples
import json
with open("train.jsonl") as f:
    lines = f.readlines()
print(f"Training examples: {len(lines)}")

# Verify format
sample = json.loads(lines[0])
assert "messages" in sample, "ERROR: train.jsonl must have 'messages' field"
assert len(sample["messages"]) >= 2, "ERROR: each example needs at least 2 messages"
print(f"Format OK: {len(sample['messages'])} messages per example")
print(f"Roles: {[m['role'] for m in sample['messages']]}")

In [ ]:
# Cell 3: Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

print(f"Model: {model.config._name_or_path}")

In [ ]:
# Cell 4: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Cell 5: Load dataset and format as text
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(f"Raw dataset: {len(raw_dataset)} examples")

# Pre-apply chat template to convert messages → text strings
# This avoids Jinja template issues with dict vs object access
def format_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = raw_dataset.map(format_to_text)
print(f"Formatted dataset: {len(dataset)} examples")
print(f"Sample (first 200 chars): {dataset[0]['text'][:200]}...")

In [ ]:
# Cell 6: Train
# Reduced batch_size and max_seq_length to fit T4 16GB VRAM
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        output_dir="hadith-scholar-output",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=25,
        save_steps=250,
        fp16=True,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=1024,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete! Loss: {stats.training_loss:.4f}")

In [ ]:
# Cell 7: Quick test
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a knowledgeable Islamic scholar assistant."},
    {"role": "user", "content": "What is the significance of the isnad in hadith verification?"},
]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
outputs = model.generate(inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Cell 8: Export to GGUF
print("Exporting to GGUF Q4_K_M (takes a few minutes)...")
model.save_pretrained_gguf(
    "hadith-scholar-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Export complete!")

import os, glob
for f in glob.glob("hadith-scholar-gguf/*"):
    size = os.path.getsize(f) / (1024**3)
    print(f"  {os.path.basename(f)}: {size:.2f} GB")

In [ ]:
# Cell 9: Download GGUF file
import glob
from google.colab import files

gguf_files = glob.glob("hadith-scholar-gguf/*.gguf")
if gguf_files:
    print(f"Downloading: {gguf_files[0]}")
    print("After download, run on your Mac:")
    print("  mv ~/Downloads/*.gguf models/hadith-scholar-q4km.gguf")
    print("  ollama create hadith-scholar -f models/Modelfile")
    print("  OLLAMA_MODEL=hadith-scholar make server")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Check export step above for errors.")